# Copernicus DEM · Pechea
Ruleaza celulele in ordine. Schimba `PROJECT_ID` cu ID-ul proiectului Google Cloud activat pentru Earth Engine. Ultima celula porneste exporturile doar daca setezi `PORNESTE_EXPORTUL = True`.

Iesiri: `dem_m.tif` (elevatie) si `slope_deg.tif` (panta), pe aceeasi grila de 10 m ca Sentinel-1, plus `summary_dem.json`.

In [ ]:
%pip -q install earthengine-api geemap
import ee
from IPython.display import display

PROJECT_ID = "ID_PROIECT"
if PROJECT_ID == "ID_PROIECT":
    raise ValueError("Completeaza PROJECT_ID cu ID-ul proiectului tau Google Cloud.")

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

## Functiile folosite
DEM-ul Copernicus GLO-30 are pixeli de 30 m. Il aducem pe grila comuna de 10 m cu interpolare bilineara, dar pastram rezolutia sursa in `summary_dem.json`. Panta se calculeaza la 30 m, inainte de reesantionare.

In [ ]:
import ee


# Ordine: longitudine vest, latitudine sud, longitudine est, latitudine nord.
PECHEA_BOUNDS = [27.61, 45.46, 27.87, 45.74]

# UTM 35N foloseste metri si e potrivit pentru Pechea (aceeasi grila ca Sentinel-1).
CRS = "EPSG:32635"
PIXEL_SIZE_M = 10

# Colectia fara "_2024_1" (COPERNICUS/DEM/GLO30) e marcata ca invechita in catalogul Earth Engine.
DEM_COLECTIE = "COPERNICUS/DEM/GLO30_2024_1"
DEM_RESOLUTIE_SURSA_M = 30


# 1. DEM-ul e o colectie de dale. Le unim si pastram proiectia nativa (30 m).
def incarca_dem(aoi):
    dale = ee.ImageCollection(DEM_COLECTIE).filterBounds(aoi)
    proiectie_nativa = dale.first().projection()
    # setDefaultProjection face ca panta sa se calculeze la 30 m, nu la scara hartii.
    return (
        dale.select("DEM").mosaic()
        .setDefaultProjection(proiectie_nativa)
        .rename("dem_m")
    )


# 2. Panta in grade, calculata pe DEM-ul la rezolutia lui nativa.
def calculeaza_panta(dem):
    return ee.Terrain.slope(dem).rename("slope_deg")


# 3. Punem imaginea pe grila comuna de 10 m (aceeasi origine ca la Sentinel-1).
def pe_grila_comuna(imagine, aoi, *, pixel_m=PIXEL_SIZE_M, crs=CRS):
    grila = [pixel_m, 0, 0, 0, -pixel_m, 0]
    # Nu taiem inainte de reesantionare: interpolarea bilineara are nevoie de pixelii
    # vecini de la margine. Pixelii NaN din colturile rasterului sunt in afara zonei:
    # dreptunghiul lon/lat e usor inclinat (~0.6 grade) in UTM 35N.
    return (
        imagine.resample("bilinear")
        .reproject(crs=crs, crsTransform=grila)
        .clip(aoi)
    )


# 4. Valorile minime si maxime din zona, la 30 m, ca sa vedem ca datele au sens.
def verifica_valori(imagine, aoi):
    return imagine.reduceRegion(
        reducer=ee.Reducer.minMax(), geometry=aoi,
        scale=DEM_RESOLUTIE_SURSA_M, maxPixels=10_000_000,
    ).getInfo()


# 5. Afisam straturile in notebook.
def afiseaza_harta(aoi, dem, panta):
    import geemap

    harta = geemap.Map()
    harta.centerObject(aoi, 12)
    harta.addLayer(
        dem, {"min": 0, "max": 150, "palette": ["333399", "00a2e5", "55dd77", "ffff99", "aa926b", "ffffff"]},
        "Elevatie (m)",
    )
    harta.addLayer(panta, {"min": 0, "max": 10, "palette": ["white", "orange", "red"]}, "Panta (grade)", False)
    return harta


# 6. Exportam o banda in Google Drive, ca GeoTIFF (aceeasi functie ca la Sentinel-1).
def exporta_geotiff(imagine, aoi, nume, *, folder="ROSPIN_Pechea",
                     pixel_m=PIXEL_SIZE_M, crs=CRS):
    # Pastram aceeasi grila la export. -9999 reprezinta pixelii fara date.
    nodata = -9999
    task = ee.batch.Export.image.toDrive(
        image=imagine.clip(aoi).toFloat().unmask(nodata, sameFootprint=False),
        description=nume,
        fileNamePrefix=nume,
        folder=folder,
        region=aoi,
        crs=crs,
        crsTransform=[pixel_m, 0, 0, 0, -pixel_m, 0],
        maxPixels=100_000_000,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True, "noData": nodata},
    )
    task.start()
    return task

## Verifica datele
Elevatia din zona de lucru trebuie sa aiba valori plauzibile pentru Pechea (zeci de metri). Cateva valori usor negative sunt normale: DEM-ul e un model de suprafata (include cladiri si vegetatie), in sistemul vertical EGM2008.

In [ ]:
aoi = ee.Geometry.Rectangle(PECHEA_BOUNDS)

dem = incarca_dem(aoi)
panta = calculeaza_panta(dem)

print("elevatie (m):", verifica_valori(dem, aoi))
print("panta (grade):", verifica_valori(panta, aoi))

# Aducem totul pe grila comuna de 10 m.
imagini = {
    "dem_m": pe_grila_comuna(dem, aoi),
    "slope_deg": pe_grila_comuna(panta, aoi),
}

display(afiseaza_harta(aoi, imagini["dem_m"], imagini["slope_deg"]))

## Export pentru aplicatie
Dupa ce verifici harta, seteaza `PORNESTE_EXPORTUL = True` si ruleaza celula o data. Earth Engine scrie rezultatele in Google Drive, in `ROSPIN_Pechea`. `summary_dem.json` se descarca din Colab cu celula de la final.

In [ ]:
import json
from pathlib import Path

PORNESTE_EXPORTUL = False
if PORNESTE_EXPORTUL:
    sumar = {
        "event_id": "pechea",
        "name": "Pechea, Galati",
        "source": DEM_COLECTIE,
        "aoi_bbox_wgs84": PECHEA_BOUNDS,
        "pixel_size_m": PIXEL_SIZE_M,
        "source_resolution_m": DEM_RESOLUTIE_SURSA_M,
        "resampling": "bilinear",
        "crs": CRS,
        "vertical_datum": "EGM2008 (EPSG:3855)",
        "model_type": "DSM: include cladiri si vegetatie, nu doar terenul gol",
        "bands": {"dem_m": "elevatie in metri", "slope_deg": "panta in grade"},
        "files": {nume: f"{nume}.tif" for nume in imagini},
        "mask_legend": {"NaN": "in afara zonei (colturile rasterului)", "-9999": "fara date"},
        "attribution": (
            "produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence "
            "and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; "
            "all rights reserved"
        ),
        "disclaimer": (
            "The organisations in charge of the Copernicus programme by law or by delegation "
            "do not incur any liability for any use of the Copernicus WorldDEM-30."
        ),
    }
    taskuri = {nume: exporta_geotiff(imagine, aoi, nume)
               for nume, imagine in imagini.items()}
    Path("summary_dem.json").write_text(
        json.dumps(sumar, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Exporturi pornite. Verifica starea lor in celula urmatoare.")
else:
    print("Exportul e oprit. Seteaza PORNESTE_EXPORTUL = True cand harta e gata.")

In [ ]:
# Ruleaza din nou celula asta pentru starea exporturilor, fara sa le repornesti.
for nume, task in globals().get("taskuri", {}).items():
    status = task.status()
    print(nume, status.get("state"), status.get("error_message", ""))

In [ ]:
from google.colab import files
from pathlib import Path

if Path("summary_dem.json").exists():
    files.download("summary_dem.json")
else:
    print("Porneste intai exportul ca sa se genereze summary_dem.json.")